# exp059_pf_model_diff_foldsafe_surface_shrink inference

Port only the selected `lgbm_capacity_pf_model_diff_foldsafe_raw` train-side candidate. Visible wells keep the public physical branch; hidden wells use PF/Beam selector features plus full-train exp052/054 source predictions and the exp059 raw LightGBM residual model.

## Contents

1. Setup and configuration
2. Selected inference candidate
3. Fit models and generate submission
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import os

from pf_model_diff_inference import generate_pf_model_diff_submission
from settings import EXPERIMENT_NAME, ExperimentPaths, load_config

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", config["experiment"].get("route"))
print("Root:", paths.root)
print("Train data:", paths.train_data_dir)
print("Test data:", paths.test_data_dir)
print("Sample submission:", paths.sample_submission_path)
print("Submission path:", paths.submission_path)
print("Artifacts:", paths.artifacts_dir)
print("Debug:", DEBUG, "Max wells:", MAX_WELLS)

## 2. Selected inference candidate

In [ ]:
print("Selected train candidate: lgbm_capacity_pf_model_diff_foldsafe_raw")
print("Inference variant:", config.get("inference", {}).get("selected_variant", "lgbm_capacity_pf_model_diff_foldsafe"))
print("Postprocess: raw")
print("Model-diff sources:")
for source in config["audit"].get("model_diff_sources", []):
    print(" -", source["name"], source.get("prediction_method"))

## 3. Fit models and generate submission

In [ ]:
summary = generate_pf_model_diff_submission(
    paths,
    config,
    max_wells=MAX_WELLS if DEBUG else MAX_WELLS,
)
print(json.dumps(summary, indent=2, sort_keys=True))

## 4. Metrics and artifacts

In [ ]:
import pandas as pd

submission = pd.read_csv(paths.submission_path)
print("submission rows:", len(submission))
print("submission exists:", paths.submission_path.exists())
for path in sorted(paths.artifacts_dir.glob("*")):
    if path.is_file():
        print(path.name, path.stat().st_size)
submission.head()